# CropSense — Crop Disease Classifier (Colab → Phone / Cloud Run CPU)

Trains a small EfficientNetV2-B0 image classifier and exports a **TFLite** model (~6–12 MB) that runs **on the phone** (offline) or on **Cloud Run CPU**.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (enough; ~1.5–2 units/hr).
2. Runtime → **Run in background** (Colab Pro).
3. Runtime → Run all. The runtime **shuts itself down** at the end.

**Data:** uses PlantVillage (38 classes) by default. PlantVillage photos are lab images on plain backgrounds, so accuracy on real field photos will be lower. For real-world accuracy, put your own field photos in Drive as `CUSTOM_DATA_DIR/<class_name>/*.jpg` and they are used instead.

**Budget:** ~2–4 hours on T4 ≈ **5–8 compute units**.

In [ ]:
# ---- Config ----
RUN_NAME = 'crop_disease_v1'
IMG_SIZE = 224
BATCH = 64
EPOCHS_HEAD = 5        # train new classifier head only
EPOCHS_FINETUNE = 10   # then unfreeze top of backbone
CUSTOM_DATA_DIR = '/content/drive/MyDrive/cropsense/data/crop_photos'  # optional: <class>/*.jpg
AUTO_SHUTDOWN = True

from google.colab import drive
drive.mount('/content/drive')

import os, json
OUT = f'/content/drive/MyDrive/cropsense/models/{RUN_NAME}'
os.makedirs(OUT, exist_ok=True)

import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
# ---- Data ----
AUTOTUNE = tf.data.AUTOTUNE

if os.path.isdir(CUSTOM_DATA_DIR) and os.listdir(CUSTOM_DATA_DIR):
    print('Using custom data from Drive:', CUSTOM_DATA_DIR)
    common = dict(image_size=(IMG_SIZE, IMG_SIZE), batch_size=None, seed=42)
    train_ds = tf.keras.utils.image_dataset_from_directory(CUSTOM_DATA_DIR, validation_split=0.2, subset='training', **common)
    rest_ds = tf.keras.utils.image_dataset_from_directory(CUSTOM_DATA_DIR, validation_split=0.2, subset='validation', **common)
    class_names = train_ds.class_names
    n_rest = int(rest_ds.cardinality())
    val_ds, test_ds = rest_ds.take(n_rest // 2), rest_ds.skip(n_rest // 2)
else:
    print('Using PlantVillage from TFDS')
    import tensorflow_datasets as tfds
    (train_ds, val_ds, test_ds), info = tfds.load(
        'plant_village', split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
        as_supervised=True, with_info=True)
    class_names = info.features['label'].names

NUM_CLASSES = len(class_names)
print(NUM_CLASSES, 'classes')
with open(f'{OUT}/labels.json', 'w') as f:
    json.dump(class_names, f, indent=1)

def resize(img, label):
    img = tf.image.resize(tf.cast(img, tf.float32), (IMG_SIZE, IMG_SIZE))
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 40.0)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    img = tf.image.random_crop(tf.image.resize(img, (IMG_SIZE + 32, IMG_SIZE + 32)), (IMG_SIZE, IMG_SIZE, 3))
    return tf.clip_by_value(img, 0.0, 255.0), label

train = train_ds.map(resize, AUTOTUNE).cache().shuffle(4096).map(augment, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
val = val_ds.map(resize, AUTOTUNE).batch(BATCH).cache().prefetch(AUTOTUNE)
test = test_ds.map(resize, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

In [ ]:
# ---- Model (expects raw 0-255 RGB input; preprocessing is built in) ----
base = tf.keras.applications.EfficientNetV2B0(
    include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
base.trainable = False

inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
model = tf.keras.Model(inputs, outputs)

# Checkpoints on Drive: resume automatically if Colab disconnects / times out.
def callbacks(stage):
    return [
        tf.keras.callbacks.BackupAndRestore(f'{OUT}/backup_{stage}'),
        tf.keras.callbacks.ModelCheckpoint(f'{OUT}/best_{stage}.keras', monitor='val_accuracy', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(f'{OUT}/log_{stage}.csv', append=True),
    ]

In [ ]:
# ---- Stage 1: train head ----
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=val, epochs=EPOCHS_HEAD, callbacks=callbacks('head'))

# ---- Stage 2: fine-tune top of backbone (BatchNorm stays frozen) ----
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=val, epochs=EPOCHS_FINETUNE, callbacks=callbacks('finetune'))

loss, acc = model.evaluate(test)
print(f'Test accuracy: {acc:.4f}')
model.save(f'{OUT}/final.keras')

In [ ]:
# ---- Export: TFLite float16 (phone / web / Cloud Run CPU) ----
tf.keras.mixed_precision.set_global_policy('float32')
export_model = tf.keras.models.clone_model(model)
export_model.set_weights(model.get_weights())
export_model.export(f'{OUT}/saved_model')

converter = tf.lite.TFLiteConverter.from_saved_model(f'{OUT}/saved_model')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite = converter.convert()
with open(f'{OUT}/crop_disease_fp16.tflite', 'wb') as f:
    f.write(tflite)

with open(f'{OUT}/model_card.json', 'w') as f:
    json.dump({'run': RUN_NAME, 'input': [1, IMG_SIZE, IMG_SIZE, 3], 'input_range': '0-255 RGB float32',
               'output': 'softmax over labels.json', 'test_accuracy': float(acc),
               'tflite_mb': round(len(tflite) / 1e6, 2)}, f, indent=1)
print('Saved to', OUT, '| TFLite MB:', round(len(tflite) / 1e6, 2))

In [ ]:
# ---- Auto-shutdown: stop burning compute units ----
if AUTO_SHUTDOWN:
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()